# ⚡ OmniVoice + Whisper GPU Worker (Google Colab T4 Miễn Phí)

Notebook này biến GPU **Tesla T4 (16 GB VRAM)** trên Google Colab thành máy chủ tăng tốc tổng hợp giọng đọc tiếng Việt **nhanh gấp ~40–50 lần CPU** cho dự án **Video Studio** trên máy tính của bạn.

### Cách dùng cực gọn:
1. Trên thanh menu Colab, kiểm tra **Runtime → Change runtime type → T4 GPU**.
2. *(Tùy chọn)* Nếu bạn có sẵn **Domain trên Cloudflare**, điền `CLOUDFLARE_TUNNEL_TOKEN`, `CUSTOM_PUBLIC_URL` và `FIXED_API_TOKEN` ở ô bên dưới để cố định link vĩnh viễn. Nếu để trống, hệ thống sẽ tự tạo link `https://xxxx.trycloudflare.com` miễn phí.
3. Bấm nút **▶ Play** ở ô code bên dưới → Đợi ~1–2 phút hiện dòng lệnh **`vstudio.bat colab --url ...`** → Copy dán vào cửa sổ dòng lệnh (hoặc gửi thẳng cho AI) ở máy tính của bạn!

In [ ]:
# @title ▶ Bấm Play để khởi chạy Máy chủ GPU OmniVoice + Cloudflare Tunnel
# @markdown ### (Tùy chọn) Cấu hình Domain Cloudflare riêng của bạn (Để trống nếu muốn tự sinh link `trycloudflare.com` miễn phí):
CLOUDFLARE_TUNNEL_TOKEN = "" # @param {type:"string"}
CUSTOM_PUBLIC_URL = "" # @param {type:"string"}
FIXED_API_TOKEN = "" # @param {type:"string"}

import os, re, secrets, subprocess, threading, time, base64, io

print('⏳ [1/4] Đang cài đặt OmniVoice, faster-whisper, FastAPI & Cloudflared...')
subprocess.run('pip install -q omnivoice faster-whisper fastapi uvicorn soundfile soxr', shell=True, check=True)
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared', shell=True, check=True)

import numpy as np
import soundfile as sf
import soxr
import torch
import uvicorn
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
from omnivoice import OmniVoice
from faster_whisper import WhisperModel

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU (Hãy đổi Runtime sang T4 GPU!)'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32
print(f'🚀 [2/4] Đang nạp OmniVoice + Whisper lên {GPU_NAME} ({DTYPE})...')

tts_model = OmniVoice.from_pretrained('splendor1811/omnivoice-vietnamese', device_map=DEVICE, torch_dtype=DTYPE)
SR = tts_model.sampling_rate
asr_model = WhisperModel('base', device=DEVICE, compute_type='float16' if DEVICE == 'cuda' else 'int8')

API_TOKEN = FIXED_API_TOKEN.strip() if FIXED_API_TOKEN.strip() else secrets.token_urlsafe(18)
app = FastAPI(title='OmniVoice Colab GPU Worker')

def trim_silence(wav, sr, thr_db=-42.0, pad_ms=35):
    peak = float(np.max(np.abs(wav)))
    if peak < 1e-6:
        return wav
    thr = peak * (10 ** (thr_db / 20.0))
    win = max(1, int(0.01 * sr))
    env = np.convolve(np.abs(wav), np.ones(win) / win, mode='same')
    idx = np.where(env > thr)[0]
    if len(idx) == 0:
        return wav
    pad = int(pad_ms * sr / 1000)
    out = wav[max(0, idx[0] - pad):min(len(wav), idx[-1] + pad)].copy()
    fade = min(int(0.015 * sr), len(out) // 4)
    if fade > 1:
        out[:fade] *= np.linspace(0, 1, fade, dtype=np.float32)
        out[-fade:] *= np.linspace(1, 0, fade, dtype=np.float32)
    return out

def synth_clauses(text, inst, spd):
    words = text.split()
    parts = [text] if len(words) <= 36 else [p.strip() for p in re.split(r'(?<=[.!?;:])\s+', text) if p.strip()]
    if len(parts) <= 1 and len(words) > 36:
        mid = len(words) // 2
        parts = [' '.join(words[:mid]), ' '.join(words[mid:])]
    chunks, pause = [], np.zeros(int(0.18 * SR), dtype=np.float32)
    for p in parts:
        out = tts_model.generate(text=p, language='vi', instruct=inst, speed=spd, pad_duration=0.02, fade_duration=0.015)
        chunks.append(trim_silence(out[0].astype(np.float32), SR))
        chunks.append(pause)
    return np.concatenate(chunks[:-1])

class SynthItem(BaseModel):
    id: str
    say: str
    speed: float = 1.08
    max_dur: float = 12.0
    instruct: str = 'male, moderate pitch'

class BatchReq(BaseModel):
    items: list[SynthItem]

@app.get('/health')
def health(x_api_token: str = Header(default='')):
    if x_api_token != API_TOKEN:
        raise HTTPException(status_code=401, detail='Invalid token')
    return {'ok': True, 'device': DEVICE, 'gpu': GPU_NAME, 'sr': SR}

@app.post('/synthesize')
def synthesize(req: BatchReq, x_api_token: str = Header(default='')):
    if x_api_token != API_TOKEN:
        raise HTTPException(status_code=401, detail='Invalid token')
    results = []
    t_all = time.time()
    for it in req.items:
        spd = it.speed
        for attempt in range(2):
            audio = synth_clauses(it.say, it.instruct, spd)
            dur = len(audio) / SR
            if dur <= it.max_dur or attempt == 1:
                break
            spd = min(1.30, spd * (dur / it.max_dur) * 1.03)
        segs, _ = asr_model.transcribe(soxr.resample(audio, SR, 16000), language='vi')
        asr_txt = ' '.join(s.text.strip() for s in segs)
        buf = io.BytesIO()
        sf.write(buf, audio, SR, format='WAV', subtype='PCM_16')
        results.append({
            'id': it.id,
            'dur': round(len(audio) / SR, 3),
            'asr': asr_txt,
            'wav_b64': base64.b64encode(buf.getvalue()).decode('ascii'),
        })
        print(f'  ✓ [{it.id}] {len(audio)/SR:.2f}s | {asr_txt}')
    print(f'⚡ Đã xử lý xong {len(results)} câu trên {GPU_NAME} chỉ trong {time.time()-t_all:.1f}s!')
    return {'ok': True, 'gpu': GPU_NAME, 'results': results}

threading.Thread(target=lambda: uvicorn.run(app, host='127.0.0.1', port=8000, log_level='warning'), daemon=True).start()
time.sleep(2)

print('🌐 [3/4] Đang mở đường hầm HTTPS Cloudflare...')
if CLOUDFLARE_TUNNEL_TOKEN.strip():
    cf = subprocess.Popen(['/usr/local/bin/cloudflared', 'tunnel', 'run', '--token', CLOUDFLARE_TUNNEL_TOKEN.strip()], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    public_url = CUSTOM_PUBLIC_URL.strip().rstrip('/') or 'https://tts.your-domain.com'
    time.sleep(4)
else:
    cf = subprocess.Popen(['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    public_url = None
    for line in cf.stdout:
        m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
        if m:
            public_url = m.group(0)
            break

print('\n' + '=' * 78)
print(f'✅ [4/4] MÁY CHỦ COLAB GPU ĐÃ SẴN SÀNG! ({GPU_NAME})')
print('👉 Copy dòng lệnh dưới đây dán vào Terminal trên máy tính (hoặc gửi cho AI):\n')
print(f'.\\vstudio.bat colab --url "{public_url}" --token "{API_TOKEN}"')
print('\n(Giữ tab Colab này mở trong lúc tạo giọng đọc; làm xong đóng tab là xong!)')
print('=' * 78 + '\n')

try:
    while True:
        time.sleep(30)
except KeyboardInterrupt:
    cf.terminate()
